# Build & Save Parquet Files
Runs once. Produces everything the modelling notebook needs.

**Outputs**
- `data/global_counts.parquet` — global lemma counts across all titles
- `data/subject_counts/XX.pkl` — per-subject (year_word, year_total) cache
- `data/word_year_subject.parquet` — flat modelling table (word × subject × year)
- `data/subject_meta.parquet` — subject code → name + title count lookup

In [35]:
# Build & Save Parquet Files
# Runs once. Produces everything the modelling notebook needs.
#
# Outputs
# - data/global_counts_v5_final.parquet
# - data/subject_counts/XX_v5_final.pkl
# - data/word_year_subject_v5_final.parquet
# - data/subject_meta_v5_final.parquet

import re
import os
import pickle
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from tqdm.auto import tqdm

# bump this any time the tokenizer, filter, or dataset changes.
# stops old cached files from silently being loaded instead of rebuilt.
PIPELINE_VERSION = "v5_final"

DATA_PATH = Path.cwd().parent.parent / "data" / "processed" / "version2_new_dataset_fitted.csv"
OUT_DIR   = Path("data")
SUBJ_DIR  = OUT_DIR / "subject_counts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBJ_DIR.mkdir(parents=True, exist_ok=True)

def versioned(path: Path) -> Path:
    return path.with_name(f"{path.stem}_{PIPELINE_VERSION}{path.suffix}")

EN_PKL      = versioned(Path("data/keyword_trends_english.pkl"))
GLOBAL_PATH = versioned(OUT_DIR / "global_counts.parquet")
BIGRAM_PATH = versioned(OUT_DIR / "bigram_model.pkl")
MODEL_PATH  = versioned(OUT_DIR / "word_year_subject.parquet")
META_PATH   = versioned(OUT_DIR / "subject_meta.parquet")

print("Paths ready")

Paths ready


In [36]:
import fasttext

raw = pd.read_csv(DATA_PATH, usecols=[
    "thesis", "year", "subject_code", "subject_name",
    "subject_prediction_confidence",
])

is_original = raw["subject_code"].notna() & raw["subject_prediction_confidence"].isna()
print(f"Total rows: {len(raw):,}")
print(f"Original-code rows: {is_original.sum():,} ({is_original.mean():.1%})")
print(f"Predicted-code rows dropped: {(~is_original & raw['subject_code'].notna()).sum():,}")

orig = raw[is_original].dropna(subset=["thesis", "year"]).copy()
orig["year"] = pd.to_numeric(orig["year"], errors="coerce")
orig = orig.dropna(subset=["year"])
orig["year"] = orig["year"].astype(int)
orig["code"] = orig["subject_code"].astype(int).astype(str).str.zfill(2)
print(f"Usable rows after cleaning: {len(orig):,} | subjects: {orig['code'].nunique()}")

if EN_PKL.exists():
    print("English cache exists, loading...")
    df_en = pd.read_pickle(EN_PKL)
else:
    model = fasttext.load_model("lid.176.bin")
    titles = (orig["thesis"].astype(str)
              .str.replace("\n", " ", regex=False).str.strip().tolist())
    labels, _ = model.predict(titles)
    orig["lang_ft"] = [l[0].replace("__label__", "") if l else "unknown" for l in labels]
    df_en = orig[orig["lang_ft"] == "en"].copy()
    df_en.to_pickle(EN_PKL)
    print(f"English titles: {len(df_en):,} ({len(df_en)/len(orig):.1%})")

df_en["year"] = df_en["year"].astype(int)
df_en["code"] = df_en["subject_code"].astype(int).astype(str).str.zfill(2)
print(f"df_en ready: {len(df_en):,} rows | {df_en['code'].nunique()} subjects")

Total rows: 338,532
Original-code rows: 202,492 (59.8%)
Predicted-code rows dropped: 77,711
Usable rows after cleaning: 160,548 | subjects: 63
English titles: 144,594 (90.1%)
df_en ready: 144,594 rows | 63 subjects


In [37]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

ACADEMIC_STOP = {
    "study", "studies", "analysis", "approach", "approaches", "method",
    "methods", "result", "results", "problem", "problems", "application",
    "applications", "using", "based", "paper", "thesis", "research",
    "theory", "case", "system", "systems", "model", "models", "general",
    "certain", "some", "new", "toward", "towards", "note", "notes",
    "essay", "essays",
}

KEEP_POS = {"NOUN", "ADJ", "VERB"}   # PROPN handled separately, kept only inside bigrams

def strip_accents(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def tokenize_doc(doc):
    return [
        token.lemma_ for token in doc
        if (token.pos_ in KEEP_POS or token.pos_ == "PROPN")
        and not token.is_stop
        and len(token.lemma_) >= 4
        and token.lemma_ not in ACADEMIC_STOP
    ]

print("spacy loaded")

spacy loaded


In [38]:
from gensim.models.phrases import Phrases, Phraser

if BIGRAM_PATH.exists():
    print("Loading bigram model...")
    with open(BIGRAM_PATH, "rb") as f:
        bigram = pickle.load(f)
else:
    print("Training bigram model on all titles...")
    all_titles_clean = [strip_accents(str(t).lower()) for t in df_en["thesis"]]

    all_token_lists = []
    for doc in tqdm(nlp.pipe(all_titles_clean, batch_size=512, n_process=1),
                    total=len(all_titles_clean), desc="tokenizing for bigram training"):
        all_token_lists.append(tokenize_doc(doc))

    phrases = Phrases(all_token_lists, min_count=30, threshold=15)
    bigram  = Phraser(phrases)

    with open(BIGRAM_PATH, "wb") as f:
        pickle.dump(bigram, f)
    print(f"Saved bigram model to {BIGRAM_PATH}")

bigram_vocab = [k.decode() if isinstance(k, bytes) else k
                for k in bigram.phrasegrams.keys()]
print(f"Bigrams learned: {len(bigram_vocab)}")
print("Sample:", bigram_vocab[:20])

def tokenize_with_bigrams(doc):
    toks = tokenize_doc(doc)
    merged = list(bigram[toks])
    propn = {t.lemma_ for t in doc if t.pos_ == "PROPN"}
    return [w for w in merged if "_" in w or w not in propn]

Training bigram model on all titles...


tokenizing for bigram training:   0%|          | 0/144594 [00:00<?, ?it/s]

Saved bigram model to data\bigram_model_v5_final.pkl
Bigrams learned: 447
Sample: ['partial_differential', 'differential_equation', 'initial_value', 'high_school', 'real_time', 'finite_difference', 'boundary_condition', 'boundary_value', 'computational_complexity', 'preservice_teacher', 'central_limit', 'limit_theorem', 'upper_bound', 'time_series', 'projective_plane', 'computer_graphic', 'analytic_function', 'asymptotic_expansion', 'optimal_control', 'kahler_manifold']


In [39]:
if GLOBAL_PATH.exists():
    print("global_counts already exists, loading...")
    _g            = pd.read_parquet(GLOBAL_PATH)
    global_counts = Counter(dict(zip(_g["lemma"], _g["count"])))
    global_total  = int(_g["count"].sum())
else:
    print("Building global counts...")
    global_counts = Counter()
    global_total  = 0

    all_titles_clean = [strip_accents(str(t).lower()) for t in df_en["thesis"]]
    for doc in tqdm(nlp.pipe(all_titles_clean, batch_size=512, n_process=1),
                    total=len(all_titles_clean), desc="global tokenize"):
        toks = tokenize_with_bigrams(doc)
        global_counts.update(toks)
        global_total += len(toks)

    pd.DataFrame(global_counts.most_common(), columns=["lemma", "count"]).to_parquet(GLOBAL_PATH, index=False)
    print(f"Saved {GLOBAL_PATH}")

print(f"Global vocabulary: {len(global_counts):,} lemmas | {global_total:,} total tokens")

Building global counts...


global tokenize:   0%|          | 0/144594 [00:00<?, ?it/s]

Saved data\global_counts_v5_final.parquet
Global vocabulary: 27,804 lemmas | 745,668 total tokens


In [40]:
all_subjects = sorted(df_en["code"].unique())
print(f"Processing {len(all_subjects)} subjects...")

for subj in tqdm(all_subjects, desc="subjects"):
    out_path = SUBJ_DIR / f"{subj}_{PIPELINE_VERSION}.pkl"
    if out_path.exists():
        continue

    sub_df       = df_en[df_en["code"] == subj].reset_index(drop=True)
    titles_clean = [strip_accents(str(t).lower()) for t in sub_df["thesis"]]
    years_list   = sub_df["year"].tolist()

    year_word, year_total = {}, {}
    for year, doc in zip(years_list, nlp.pipe(titles_clean, batch_size=512, n_process=1)):
        toks = tokenize_with_bigrams(doc)
        year_word.setdefault(year, Counter()).update(toks)
        year_total[year] = year_total.get(year, 0) + len(toks)

    with open(out_path, "wb") as f:
        pickle.dump({"year_word": year_word, "year_total": year_total}, f)

print("All subject counts saved.")

Processing 63 subjects...


subjects:   0%|          | 0/63 [00:00<?, ?it/s]

All subject counts saved.


In [41]:
# %% [3. flat modelling table]
if MODEL_PATH.exists():
    print("word_year_subject already exists, skipping.")
else:
    records = []

    for subj in tqdm(all_subjects, desc="building flat table"):
        pkl_path = SUBJ_DIR / f"{subj}_{PIPELINE_VERSION}.pkl"
        if not pkl_path.exists():
            continue

        with open(pkl_path, "rb") as f:
            counts = pickle.load(f)

        year_word        = counts["year_word"]
        year_total       = counts["year_total"]
        subj_total_words = sum(year_total.values())

        for year, word_counts in year_word.items():
            total = year_total[year]
            for word, count in word_counts.items():
                subject_share = count / max(subj_total_words, 1)
                global_share  = global_counts.get(word, 0) / max(global_total, 1)  # patched
                specificity   = subject_share / max(global_share, 1e-9)
                records.append({
                    "word": word, "subject": subj, "year": year,
                    "count": count, "total_title_words": total,
                    "share": count / max(total, 1),
                    "specificity": specificity,
                })

    df_model = pd.DataFrame(records)
    df_model["year"]    = df_model["year"].astype(int)
    df_model["subject"] = df_model["subject"].astype(str).str.zfill(2)
    df_model["count"]   = df_model["count"].astype(int)

    df_model.to_parquet(MODEL_PATH, index=False)
    print(f"Saved {MODEL_PATH} - {len(df_model):,} rows, {df_model.shape[1]} columns")

building flat table:   0%|          | 0/63 [00:00<?, ?it/s]

Saved data\word_year_subject_v5_final.parquet - 445,666 rows, 7 columns


In [42]:
# ── 4. SUBJECT METADATA ────────────────────────────────────────────────────
subject_meta = (
    df_en.dropna(subset=["subject_name"])
    .groupby("code")
    .agg(subject_name=("subject_name", lambda x: x.mode()[0]), n_titles=("thesis", "count"))
    .reset_index()
    .rename(columns={"code": "subject"})
)

subject_meta.to_parquet(META_PATH, index=False)
print(f"Saved {META_PATH}")
print(subject_meta.sort_values("n_titles", ascending=False).head(10).to_string(index=False))

Saved data\subject_meta_v5_final.parquet
subject                                           subject_name  n_titles
     68                                       Computer science     23692
     91 Game theory, economics, social and behavioral sciences     17700
     62                                             Statistics     14502
     65                                     Numerical analysis      5432
     60            Probability theory and stochastic processes      5231
     90          Operations research, mathematical programming      4756
     94                Information and communication, circuits      4437
     35                         Partial differential equations      4426
     05                                          Combinatorics      3751
     11                                          Number theory      3645


In [43]:
# ── 5. SANITY CHECK ────────────────────────────────────────────────────────
df_check = pd.read_parquet(MODEL_PATH)

print("Schema:"); print(df_check.dtypes)
print(f"\nRows:     {len(df_check):,}")
print(f"Subjects: {df_check['subject'].nunique()}")
print(f"Words:    {df_check['word'].nunique():,}")
print(f"Years:    {df_check['year'].min()} - {df_check['year'].max()}")

bigram_rows = df_check[df_check["word"].str.contains("_")]
print(f"\nBigram tokens: {bigram_rows['word'].nunique():,} unique")
print("Top bigrams by count:")
print(bigram_rows.groupby("word")["count"].sum().nlargest(20).to_string())

check_words = ["network", "stochastic", "markov"]  # words that should survive filtering
for word in check_words:
    n = df_check[df_check["word"] == word]["count"].sum()
    print(f"{word}: total count = {n} {'(REMOVED, check filter)' if n == 0 else ''}")

Schema:
word                  object
subject               object
year                   int64
count                  int64
total_title_words      int64
share                float64
specificity          float64
dtype: object

Rows:     445,666
Subjects: 63
Words:    27,804
Years:    1747 - 2026

Bigram tokens: 870 unique
Top bigrams by count:
word
differential_equation    1002
time_series               758
high_dimensional          751
partial_differential      719
large_scale               713
machine_learning          602
finite_element            588
high_order                540
optimal_control           519
boundary_value            459
united_states             451
neural_network            447
banach_space              425
real_time                 397
monte_carlo               383
labor_market              330
numerical_solution        328
statistical_inference     323
markov_chain              296
second_order              295
network: total count = 4135 
stochastic: total cou

In [44]:
# %% [verify all saved outputs exist and look right]
def check_parquet(path, name):
    if not Path(path).exists():
        print(f"{name}: MISSING at {path}")
        return None
    df = pd.read_parquet(path)
    print(f"{name}: OK | rows={len(df):,} | cols={list(df.columns)}")
    print(df.head(3).to_string(index=False))
    print()
    return df

def check_pickle(path, name):
    if not Path(path).exists():
        print(f"{name}: MISSING at {path}")
        return None
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print(f"{name}: OK | type={type(obj)}")
    if isinstance(obj, dict):
        print(f"  keys: {list(obj.keys())[:5]}")
    print()
    return obj

_ = check_parquet(GLOBAL_PATH, "global_counts")
_ = check_parquet(MODEL_PATH, "word_year_subject")
_ = check_parquet(META_PATH, "subject_meta")
_ = check_pickle(BIGRAM_PATH, "bigram_model")
_ = check_pickle(EN_PKL, "english_filtered_titles")

subj_files = list(SUBJ_DIR.glob(f"*_{PIPELINE_VERSION}.pkl"))
print(f"subject_counts pkls: {len(subj_files)} found")
print("sample:", [f.name for f in subj_files[:5]])

global_counts: OK | rows=27,804 | cols=['lemma', 'count']
   lemma  count
   datum   4844
equation   4818
   group   4608

word_year_subject: OK | rows=445,666 | cols=['word', 'subject', 'year', 'count', 'total_title_words', 'share', 'specificity']
      word subject  year  count  total_title_words  share  specificity
     error      00  1981      1                  4   0.25     0.292179
  recovery      00  1981      1                  4   0.25     0.731740
concurrent      00  1981      1                  4   0.25     0.889104

subject_meta: OK | rows=63 | cols=['subject', 'subject_name', 'n_titles']
subject                       subject_name  n_titles
     00                            General       792
     01              History and biography       266
     03 Mathematical logic and foundations      2746

bigram_model: OK | type=<class 'gensim.models.phrases.FrozenPhrases'>

english_filtered_titles: OK | type=<class 'pandas.core.frame.DataFrame'>

subject_counts pkls: 63 found
samp